In [ ]:
import numpy as np 
import itertools
from matplotlib import pylab as plt


class RBF_network:
    def __init__(self, centers):
        """
        centers: массив центров RBF-нейронов
        Инициализируются веса: v0 для смещения и v1...vJ равны 0.
        """
        self.centers = centers
        self.J = centers.shape[0] # число строк в массиве centers
        self.weights = np.zeros(self.J + 1)  # [v0, v1, ..., v_J]
        # self.weights = np.ones(self.J + 1)  # [v0, v1, ..., v_J]
    
    def rbf(self, x, center):
        """
        Гауссова RBF: φ(x, c) = exp( - ||x - c||² )
        """
        return np.exp(-np.sum((x - center)**2))
    
    def compute_phi(self, x):
        """
        Вычисляет вектор выходов скрытого слоя для входного вектора x.
        Вектор имеет вид: [1, φ₁(x), φ₂(x), ..., φ_J(x)],
        где первая компонента – вход смещения (φ₀ = 1).
        """
        phi_hidden = np.array([self.rbf(x, c) for c in self.centers])
        return np.concatenate(([1], phi_hidden))
    
    def predict(self, x):
        """
        Вычисляет суммарный вход (net) и возвращает выход:
        """
        phi = self.compute_phi(x)
        net = np.dot(self.weights, phi)
        return 1 if net >= 0 else 0
    
    def learn(self, inputs, outputs, eta=0.3, max_epochs=100, show=False):
        if show:
            epoch = 0
            error_history = []
            details = []  # Итоговая таблица обучения
            steps_details = []  # Шаги по эпохам
            
            while epoch < max_epochs:
                epoch_error = 0
                weights_before_epoch = np.copy(self.weights)
                predictions = []
                epoch_steps = []  # Список шагов внутри эпохи

                # Проходим по всем обучающим примерам
                for step, (x, target) in enumerate(zip(inputs, outputs), start=1):
                    phi = self.compute_phi(x)
                    
                    weights_before_update = np.copy(self.weights)  # Веса до корректировки
                    
                    output = self.predict(x)
                    predictions.append(output)
                    error = target - output
                    epoch_error += abs(error)

                    # Коррекция весов по правилу Видроу-Хоффа
                    weight_correction = eta * error * phi
                    self.weights += weight_correction

                    # Логируем шаг
                    epoch_steps.append([
                        step, x.tolist(), np.round(weights_before_update, 3).tolist(), target, 
                        output, np.round(self.weights, 3).tolist(), epoch_error
                    ])

                # Сохраняем данные эпохи в итоговую таблицу
                details.append([epoch+1, np.round(weights_before_epoch, 3), outputs, predictions, epoch_error])
                steps_details.append((epoch+1, epoch_steps))

                error_history.append(epoch_error)
                epoch += 1

                if epoch_error == 0:
                    break

            return error_history, details, steps_details
        
        else:
            epoch = 0
            while epoch < max_epochs:
                epoch_error = 0
                predictions = []
                # Проходим по всем обучающим примерам
                for x, target in zip(inputs, outputs):
                    phi = self.compute_phi(x)
                    net = np.dot(self.weights, phi)
                    output = 1 if net >= 0 else 0
                    predictions.append(output)
                    error = target - output
                    epoch_error += abs(error)
                    # Обновление весов по правилу Видроу-Хоффа
                    self.weights += eta * error * phi
                epoch += 1
                if epoch_error == 0:
                    break
            return epoch_error           
    
    def check_network(self, inputs, outputs):
        """
        Проверяет сеть на обучающей выборке, возвращая суммарную ошибку и результаты.
        """
        total_error = 0
        for x, target in zip(inputs, outputs):
            phi = self.compute_phi(x)
            net = np.dot(self.weights, phi)
            output = 1 if net >= 0 else 0
            total_error += abs(target - output)
        return total_error


# Функции для форматирования
def format_vector(vec):
    """Преобразует numpy-массив в строку с фиксированным форматом элементов."""
    return "[" + ", ".join("{:>6.3f}".format(x) for x in vec) + "]"

def format_list(lst):
    """Преобразует список целых чисел в строку."""
    return "[" + ", ".join("{:d}".format(x) for x in lst) + "]"


In [ ]:
# Генерация полной таблицы истинности
def bool_func(x):
    return not(not( x[0] or x[1] ) or x[2] or x[3])
    # return (x[0] or x[1] or x[2]) and (x[1] or x[2] or x[3])
    # return not(( ( x[0] or x[1] ) and x[2] ) or (x[2] and x[3]))
    # return ((not(x[0]) or x[2]) and x[1]) or (x[1] and x[3])

# Генерация всех 16 возможных входных векторов длины 4
full_inputs = []
full_outputs = []
for i in range(16):
    # Формируем бинарное представление длиной 4, дополняем ведущими нулями
    inp = list(map(int, bin(i)[2:].rjust(4, '0')))
    full_inputs.append(inp)
    full_outputs.append(bool_func(inp))

full_inputs = np.array(full_inputs)
full_outputs = np.array(full_outputs)

# Вывод таблицы истинности
print("Таблица истинности:")
for x, y in zip(full_inputs, full_outputs):
    print(x, int(y))


In [ ]:
# Определение центров RBF-нейронов
# Согласно условию, число RBF-нейронов выбирается как J = min{J0, J1},
# где J0, J1 – число векторов с выходом 0 и 1 соответственно.

#===============================

positive_indices = np.where(full_outputs == 1)[0]
negative_indices = np.where(full_outputs == 0)[0]
if len(positive_indices) > len(negative_indices):
    rbf_neurons_values = full_inputs[negative_indices]
else:
    rbf_neurons_values = full_inputs[positive_indices]
centers = np.unique(rbf_neurons_values, axis=0)
print("\nЦентры RBF-нейронов:\n")
for i in range(len(centers)):
    print(f"C{i+1} = {centers[i]}\n")



In [ ]:
# Определение минимального набора обучающих векторов
#Функция для нахождения минимальных входных данных для сети, достаточных для обучения
def find_min_inputs(full_inputs, full_outputs, centers, network):
        for i in range(1,len(full_inputs)): # Перебор подмножеств от 1 до 15
            new_inputs = itertools.combinations(range(16), i)
            # перебор всех возможных сочетаний индексов входов длиной i 
            for sample in new_inputs: 
                inputs=[]
                outputs=[]
                for number in sample:
                    # формируем двоичное представление числа (4-битное)
                    inputs.append(list(map(int,bin(number)[2:].rjust(4,'0'))))
                for bin_number in inputs:
                    # генерируем выходы для поднабора входов через булеву функцию
                    outputs.append(bool_func(bin_number))
                inputs = np.array(inputs)
                outputs = np.array(outputs)

                rbf_netw = network(centers)
                # Обучение сети на выбранном подмножестве
                E = rbf_netw.learn(inputs, outputs) 
                # Расстояние Хэмминга при тестировании на полном наборе
                E_full = int(rbf_netw.check_network(full_inputs, full_outputs))
                print("E_min: ",E,"E_full: ",E_full)
                # Проверка, способна ли сеть корректно предсказывать на множестве
                if E == 0 and E_full == 0:
                    min_input_param={'weights': rbf_netw.weights,'inputs': inputs, 'outputs':outputs}
                    return min_input_param
            
min_input_param = find_min_inputs(full_inputs, full_outputs, centers, RBF_network) 
min_weights = np.round(min_input_param['weights'],3)
min_input = min_input_param['inputs']
min_output = np.array(min_input_param['outputs'], dtype = int)

print("Веса:", ", ".join(f"{w:.3f}" for w in min_weights))
print('Минимальная выборка:', min_input)
# # print('Ожидаемый выход:', min_output)

In [ ]:
# Создаем экземпляр RBF-сети с выбранными центрами
rbf_net = RBF_network(centers)

# Обучаем сеть на минимальном наборе обучающих векторов
error_history, details, steps_details = rbf_net.learn(min_input, min_output, eta=0.3, max_epochs=100, show=True)
# error_history, details, steps_details = rbf_net.learn(full_inputs, full_outputs, eta=0.3, max_epochs=100, show=True)

# E_full = rbf_net.check_network(full_inputs,full_outputs)
# print(f"Ошибка на полной выборке: {E_full}")

# Вывод таблицы обучения
print("\nТаблица обучения:")
header = "{:<8} | {:<35} | {:<25} | {:<25} | {:<15}".format("Эпоха", "Вектор весов", "Ожидаемый вектор", "Выходной вектор", "Сумм. ошибка")
# header = "{:<8} | {:<35} | {:<48} | {:<48} | {:<15}".format("Эпоха", "Вектор весов", "Ожидаемый вектор", "Выходной вектор", "Сумм. ошибка")
print(header)
print("-" * len(header))
for row in details:
    epoch, weights, expected_vector, output_vector, error = row
    weights_str = format_vector(weights)
    expected_str = format_list(expected_vector)
    output_str = format_list(output_vector)
    print("{:<8} | {:<35} | {:<25} | {:<25} | {:<15}".format(epoch, weights_str, expected_str, output_str, error))

# График ошибки

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(error_history)+1), error_history, marker='o')
plt.xlabel("Эпоха обучения")
plt.ylabel("Суммарная ошибка")
plt.title("Зависимость ошибки от числа эпох")
plt.grid(True)
plt.show()

print("\nПодробные шаги обучения по эпохам:")
for epoch_num, steps in steps_details:
    print(f"\nЭпоха #{epoch_num}:")
    print("{:<8} | {:<18} | {:<35} | {:<10} | {:<10} | {:<32} | {:<15}".format(
        "Шаг", "Входной вектор", "Веса до корр.", "Ожидание", "Прогноз", "Коррекция весов", "Суммарная ошибка"
    ))
    print("-" * 130)
    
    for step, x, w_before, expected, y, correction, err_sum in steps:
        x_str = str(x)
        w_before_str = format_vector(np.array(w_before))
        correction_str = format_vector(np.array(correction))
        print("{:<8} | {:<18} | {:<35} | {:<10} | {:<10} | {:<20} | {:<15}".format(
            step, x_str, w_before_str, expected, y, correction_str, err_sum
        ))
#####################################################
#####################################################
